# IRL 가중치 학습 테스트 - `pm_safeline.irl`

PM 세이프라인 비용함수 `cost(edge) = w1*거리 + w2*차도 + w3*전환 + w4*교차로 + w5*버스겹침`
의 가중치 w1~w5 를 **Bradley-Terry 랭킹 로스** 기반 IRL 로 학습한다(PROJECT.md §4.2, §4.4).

- `irl.route_risk` : edge 위험확률 -> route 위험도 집계 (hazard-rate 생존모델, §4.5-3)
- `irl.bradley_terry` : 경로쌍 선호 라벨 -> w1~w5 학습 (convex, `w>=0` 제약)

teacher(로드뷰 이미지 위험도 모델)는 아직 학습 전(torch 환경 준비 중)이므로, 이 노트북은
**합성(synthetic) 데이터**로 전체 IRL 흐름을 시연하고 "알려진 참값 w" 를 복원할 수 있는지
검증한다. 마지막 셀에서 실제 teacher 로 교체하는 방법을 설명한다.

> 이 스테이지는 torch 불필요 - numpy/scipy/scikit-learn 만으로 동작.

In [1]:
import numpy as np
import pandas as pd

from pm_safeline.utils.irl import (
    aggregate, route_risk, route_features,
    make_preference_pairs, fit_bradley_terry, learn_weights_from_routes,
    BTResult,
)


## 1. Route risk 집계 - hazard-rate 생존모델

`route_risk = 1 - exp(-Σ h_i*L_i) = 1 - Π(1-p_i)` (§4.5-3 권장안). 간단한 예로
집계 방법(hazard/mean/max/sum/count)을 비교한다: 위험구간이 몰려있는 경로 vs 고르게
낮은 경로가 같은 평균이어도 hazard 방법은 다르게 평가해야 한다.

In [2]:
# 위험이 한 구간에 몰린 경로 vs 고르게 낮은 경로 (평균은 비슷하게 설계)
concentrated = np.array([0.30, 0.01, 0.01, 0.01, 0.01])
spread = np.array([0.07, 0.07, 0.07, 0.06, 0.07])
print("평균:", concentrated.mean().round(4), spread.mean().round(4))

methods = ["hazard", "mean", "max", "sum", "count"]
rows = []
for name, probs in [("concentrated(위험 몰림)", concentrated), ("spread(고르게 분산)", spread)]:
    rows.append({"경로": name, **{m: round(aggregate(probs, method=m, tau=0.05), 4) for m in methods}})
pd.DataFrame(rows)

평균: 0.068 0.068


,경로,hazard,mean,max,sum,count
0,concentrated(위험 몰림),0.3276,0.068,0.30,0.34,1.0
1,spread(고르게 분산),0.2968,0.068,0.07,0.34,5.0


## 2. 합성 데이터 생성 - "참값" w 를 알고 있는 상태에서 복원 검증

- route feature = [거리, 차도, 전환, 교차로, 버스겹침] (5차원, 양수)
- true_w 로 정의한 "실제" 위험 성향에 비례해 각 route 의 edge 위험확률(p_i)을 합성
- route_risk() 로 hazard-rate 집계 -> 이게 "teacher가 준 것처럼" 취급하는 라벨 소스

In [3]:
rng = np.random.default_rng(0)

# 참값(kt.dinjae.pm_safeline.PmCostWeights 형태로 서버에 주입될 값이라 가정)
true_w = np.array([1.0, 0.5, 0.3, 0.8, 0.2])
FEATURE_NAMES = ["distance(w1)", "arterial(w2)", "transition(w3)", "crossing(w4)", "bus(w5)"]

n_routes = 300
feats = np.stack([
    route_features(
        distance_km=d, arterial=a, transition_count=t, crossing_count=c, bus_overlap=b,
    )
    for d, a, t, c, b in zip(
        rng.uniform(1, 10, n_routes),
        rng.uniform(0, 5, n_routes),
        rng.uniform(0, 8, n_routes),
        rng.uniform(0, 6, n_routes),
        rng.uniform(0, 3, n_routes),
    )
])

n_edges = 20
risks = []
for f in feats:
    # true_w 에 비례한 "실제 위험 성향" score -> edge 평균 사고확률로 변환(teacher 모델 대역)
    score = float(f @ true_w)
    mean_p = np.clip(0.01 + 0.003 * score, 0.001, 0.08)
    edge_probs = np.clip(rng.normal(mean_p, mean_p * 0.05, n_edges), 0, 0.2)
    risks.append(route_risk(edge_probs))
risks = np.array(risks)

print(f"routes: {len(feats)}, route_risk 범위: [{risks.min():.4f}, {risks.max():.4f}]")
pd.DataFrame(feats, columns=FEATURE_NAMES).assign(route_risk=risks).head()

routes: 300, route_risk 범위: [0.3354, 0.7480]


,distance(w1),arterial(w2),transition(w3),crossing(w4),bus(w5),route_risk
0,6.732655,4.484006,0.049415,5.645862,0.922550,0.647939
1,3.428080,2.916600,5.769326,3.303647,1.122381,0.540781
2,1.368762,0.201091,5.412836,5.529708,2.085801,0.498138
3,1.148749,3.557434,5.255212,2.019354,0.950550,0.447322
4,8.319432,2.845129,5.499320,4.585919,1.588966,0.687053


## 3. 선호쌍 샘플링 + Bradley-Terry 학습

`make_preference_pairs` 로 route 쌍을 뽑고 `route_risk` 가 더 낮은 쪽을 선호(prefer=1)로
라벨링한 뒤, `fit_bradley_terry` 로 `w>=0` 제약 하 convex 최적화를 수행한다.

In [4]:
X_A, X_B, prefer = make_preference_pairs(feats, risks, n_pairs=1500, seed=1)
print(f"선호쌍: {len(prefer)}개, A 선호 비율: {prefer.mean():.3f}")

result: BTResult = fit_bradley_terry(X_A=X_A, X_B=X_B, prefer=prefer, nonneg=True, l2=1e-4)
print(f"converged={result.converged}, n_pairs={result.n_pairs}, loss={result.loss:.3f}")
print("learned weights:", np.round(result.weights, 4))

선호쌍: 1500개, A 선호 비율: 0.502
converged=True, n_pairs=1500, loss=50.764
learned weights: [8.7975 4.5885 2.6114 7.248  1.5555]


## 4. 검증 - 복원된 w ≈ 참값 w (부호 + 상대적 크기)

절대 스케일은 L2 정칙화·선호쌍 노이즈에 따라 달라질 수 있으므로(로지스틱 회귀의 통상적 특성),
**부호(모두 양수)** 와 **정규화된 상대 비율**로 비교한다.

In [5]:
comparison = pd.DataFrame({
    "feature": FEATURE_NAMES,
    "true_w": true_w,
    "learned_w": np.round(result.weights, 4),
    "true_w (정규화)": np.round(true_w / true_w.sum(), 3),
    "learned_w (정규화)": np.round(result.weights / result.weights.sum(), 3),
})
corr = np.corrcoef(true_w, result.weights)[0, 1]
print(comparison.to_string(index=False))
print(f"\n상관계수(true_w vs learned_w): {corr:.4f}")
print(f"모두 음이 아님(w>=0): {bool(np.all(result.weights >= -1e-9))}")

assert corr > 0.9, "복원된 가중치가 참값과 충분히 상관되지 않음"
assert np.all(result.weights >= -1e-9), "nonneg 제약 위반"
print("\n검증 통과: 합성 데이터에서 Bradley-Terry 로 참값 w 를 근사 복원함.")

       feature  true_w  learned_w  true_w (정규화)  learned_w (정규화)
  distance(w1)     1.0     8.7975         0.357            0.355
  arterial(w2)     0.5     4.5885         0.179            0.185
transition(w3)     0.3     2.6114         0.107            0.105
  crossing(w4)     0.8     7.2480         0.286            0.292
       bus(w5)     0.2     1.5555         0.071            0.063

상관계수(true_w vs learned_w): 0.9989
모두 음이 아님(w>=0): True

검증 통과: 합성 데이터에서 Bradley-Terry 로 참값 w 를 근사 복원함.


## 5. 실제 teacher 로 교체하는 방법

이 노트북에서 `edge_probs`(합성)는 실제로는 **파인튜닝된 이미지 위험도 teacher 모델**의
출력으로 대체된다(PROJECT.md §4.4):

```python
# (실서비스 파이프라인 - teacher 모델 학습 완료 후)
# 1) 대전 전체 OSM edge 에 대해 로드뷰 이미지 수집 (pm_safeline.datasets.roadview)
# 2) teacher.predict_proba(image) -> edge_probs (사고확률)
# 3) irl.route_risk.route_risk(edge_probs, edge_lengths) 로 route 단위 위험도 집계
# 4) 실제 route 후보(예: Yen's k-최단경로로 뽑은 top-k)들에 대해
#    irl.bradley_terry.make_preference_pairs(routes, route_risks, n_pairs, seed) 로 선호쌍 생성
# 5) irl.bradley_terry.fit_bradley_terry(...) 로 최종 w1~w5 학습
# 6) 학습된 w 를 GraphHopper custom Weighting(Kotlin, PmCostWeights)에 주입 후 teacher 는 완전히 손을 뗌
```

`route_features()` 의 5개 인자(distance_km, arterial, transition_count, crossing_count,
bus_overlap)만 실제 OSM/경로 지표로 채우면, 이 노트북의 3~4단계 코드는 그대로 재사용 가능.

## 6. Kotlin 서버 주입용 출력 - `PmCostWeights(w1..w5)`

In [6]:
w1, w2, w3, w4, w5 = result.weights
print("학습된 비용함수 가중치 (PmCostWeights 주입용):")
print(f"  w1 (거리)       = {w1:.4f}")
print(f"  w2 (차도)       = {w2:.4f}")
print(f"  w3 (전환)       = {w3:.4f}")
print(f"  w4 (교차로)     = {w4:.4f}")
print(f"  w5 (버스겹침)   = {w5:.4f}")
print()
print("Kotlin data class 리터럴 예시:")
print(f"PmCostWeights(w1={w1:.4f}, w2={w2:.4f}, w3={w3:.4f}, w4={w4:.4f}, w5={w5:.4f})")

학습된 비용함수 가중치 (PmCostWeights 주입용):
  w1 (거리)       = 8.7975
  w2 (차도)       = 4.5885
  w3 (전환)       = 2.6114
  w4 (교차로)     = 7.2480
  w5 (버스겹침)   = 1.5555

Kotlin data class 리터럴 예시:
PmCostWeights(w1=8.7975, w2=4.5885, w3=2.6114, w4=7.2480, w5=1.5555)
